# Validate retrieval against original videos

This notebook uses the public `VideoRetriever` API for every image request. Read-only SQL discovers records; a diagnostic PyAV pass counts actual decoded frames and records presentation timestamps (PTS), without converting or retaining images. Nothing writes to the database or exports images.

Run with the backend environment, from the repository root, `backend`, or `backend/notebooks`. Run all cells in order. Keep the checked-in notebook output-free: do not save executed outputs. The figures below support a human check of face alignment; they do not verify identity.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
backend_dir = next(
    (candidate for parent in (cwd, *cwd.parents)
     for candidate in (parent, parent / 'backend')
     if (candidate / 'analysis_db' / 'paths.py').is_file()),
    None,
)
if backend_dir is None:
    raise RuntimeError('Launch from the repository root, backend, or backend/notebooks.')
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from contextlib import closing
from dataclasses import asdict
import math
import random
import sqlite3

import av
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from IPython.display import display, Markdown

from analysis_db.paths import DEFAULT_DB, REPOSITORY_ROOT
from analysis_db.storage import connect_database, table_counts
from video_retrieval import VideoRetriever, DetectionInfo


## Configuration and runtime policy

Set `DB_PATH` to an existing SQLite path, or leave it as `None` for `DEFAULT_DB`. Explicit relative database paths resolve against the current working directory. Original video paths resolve against `REPOSITORY_ROOT`.

The default pool contains up to three shortest originals with positive stored duration at most 120 seconds, existing files, and usable detection boxes. Metadata duration is a runtime filter only: true frame counts come from decoding to EOF. Unknown-duration and long originals are excluded by default, so the 110-minute original is not scanned. Set `VALIDATION_VIDEO_IDS` to an explicit tuple to opt into particular videos, including long or unknown-duration ones. That override bypasses the duration limit but still caps the number of videos at `MAX_VIDEOS`. Sequential decoding of a long original can take substantial time.

A fixed seed selects from sorted, valid detection IDs in the chosen videos after their true bounds are known. Boundary coverage is limited to this displayed pool.

In [ ]:
DB_PATH = None  # Example: Path('/absolute/path/to/analysis.sqlite')
SEED = 42
MAX_DURATION_SECONDS = 120.0
MAX_VIDEOS = 3
VALIDATION_VIDEO_IDS = None  # Example: (2, 4); explicit expensive-video opt-in

if MAX_VIDEOS < 1 or MAX_DURATION_SECONDS <= 0:
    raise ValueError('MAX_VIDEOS and MAX_DURATION_SECONDS must be positive.')
retriever = VideoRetriever(DB_PATH)
print(f'Repository: {REPOSITORY_ROOT}')
print(f'Database: {retriever.db_path} (default: {DEFAULT_DB})')
print(f'Working directory: {cwd}')
print(f'Selection policy: seed={SEED}; max videos={MAX_VIDEOS}; '
      f'duration limit={MAX_DURATION_SECONDS}s; explicit IDs={VALIDATION_VIDEO_IDS}')


## Database totals and candidate discovery

Connections use `read_only=True` and close after each diagnostic query. The totals are database-wide; the validation pool below is deliberately smaller. An empty or unsuitable pool stops with an actionable message.

In [ ]:
with closing(connect_database(retriever.db_path, read_only=True)) as connection:
    totals = table_counts(connection)
    display({name: totals[name] for name in ('Video', 'Frame', 'Person', 'FaceDetection')})
    connection.row_factory = sqlite3.Row
    videos = [dict(row) for row in connection.execute(
        'SELECT video_id, title, filename, file_path, duration_seconds, fps, width, height '
        'FROM Video ORDER BY video_id'
    )]

if not totals['FaceDetection']:
    raise RuntimeError('No face detections are stored in this database; ingest data first.')

explicit_ids = None if VALIDATION_VIDEO_IDS is None else set(VALIDATION_VIDEO_IDS)
eligible_videos = []
for video in videos:
    video['path'] = (REPOSITORY_ROOT / video['file_path']).resolve()
    duration = video['duration_seconds']
    short = duration is not None and math.isfinite(duration) and 0 < duration <= MAX_DURATION_SECONDS
    requested = short if explicit_ids is None else video['video_id'] in explicit_ids
    if requested:
        if video['path'].is_file():
            eligible_videos.append(video)
        else:
            print(f"Skipping missing original for video {video['video_id']}: {video['path']}")
if explicit_ids is not None:
    unknown = explicit_ids - {video['video_id'] for video in videos}
    if unknown:
        print(f'Explicit video IDs absent from database: {sorted(unknown)}')
eligible_videos.sort(key=lambda video: (
    video['duration_seconds'] if video['duration_seconds'] is not None else math.inf,
    video['video_id'],
))

# Diagnostic discovery only; actual metadata/images come from the public API.
video_by_id = {}
candidate_rows = []
with closing(connect_database(retriever.db_path, read_only=True)) as connection:
    connection.row_factory = sqlite3.Row
    for video in eligible_videos:
        rows = [dict(row) for row in connection.execute(
            'SELECT d.detection_id, f.video_id, f.frame_number, f.source_time_seconds, '
            'd.x1, d.y1, d.x2, d.y2 FROM FaceDetection d '
            'JOIN Frame f ON f.frame_id = d.frame_id WHERE f.video_id = ? '
            'ORDER BY d.detection_id', (video['video_id'],)
        )]
        usable = []
        for row in rows:
            box = tuple(row[key] for key in ('x1', 'y1', 'x2', 'y2'))
            if not all(math.isfinite(value) for value in box):
                continue
            x1, y1, x2, y2 = box
            if x1 >= x2 or y1 >= y2 or x2 <= 0 or y2 <= 0:
                continue
            if video['width'] and x1 >= video['width']:
                continue
            if video['height'] and y1 >= video['height']:
                continue
            usable.append(row)
        if usable:
            video_by_id[video['video_id']] = video
            candidate_rows.extend(usable)
            if len(video_by_id) == MAX_VIDEOS:
                break

if not candidate_rows:
    raise RuntimeError(
        'No usable detections with existing originals match the runtime policy. '
        'Check paths, choose VALIDATION_VIDEO_IDS, or increase MAX_DURATION_SECONDS.'
    )
print(f'Validation pool: {len(candidate_rows)} candidate detections in {len(video_by_id)} videos.')
display([{key: video[key] for key in (
    'video_id', 'title', 'path', 'duration_seconds', 'fps', 'width', 'height'
)} for video in video_by_id.values()])


## True decoded bounds and original presentation timestamps

This diagnostic pass enumerates the first video stream in presentation order, from zero to EOF. It records only counts, dimensions, rates, and PTS values for relevant indices. It never calls `to_ndarray`, stores decoded images, uses approximate seeking, or infers bounds from duration × FPS / `stream.frames`. The small in-memory metadata dictionary avoids rescanning each original for boundary selection.

In [ ]:
def scan_original_metadata(path, wanted_indices):
    timestamps = {}
    count = 0
    first_pts = last_pts = None
    dimensions = None
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        rate = float(stream.average_rate) if stream.average_rate else None
        for index, decoded in enumerate(container.decode(video=0)):
            pts_seconds = (float(decoded.pts * decoded.time_base)
                           if decoded.pts is not None and decoded.time_base is not None else None)
            if index == 0:
                first_pts = pts_seconds
                dimensions = (decoded.height, decoded.width)
            if index in wanted_indices or index in (0, 1):
                timestamps[index] = pts_seconds
            last_pts = pts_seconds
            count = index + 1
            del decoded
    if not count:
        raise RuntimeError(f'Original has no decodable video frames: {path}')
    timestamps[count - 1] = last_pts
    return dict(frame_count=count, height=dimensions[0], width=dimensions[1],
                average_rate=rate, first_pts=first_pts, timestamps=timestamps)

diagnostics = {}
for video_id, video in video_by_id.items():
    wanted = {row['frame_number'] for row in candidate_rows if row['video_id'] == video_id}
    print(f"Scanning original video {video_id}: {video['title']}")
    diagnostic = scan_original_metadata(video['path'], wanted)
    diagnostics[video_id] = diagnostic
    display({key: value for key, value in diagnostic.items() if key != 'timestamps'})

# Recheck eligibility against actual decoded dimensions and bounds.
valid_rows = []
for row in candidate_rows:
    diagnostic = diagnostics[row['video_id']]
    if (0 <= row['frame_number'] < diagnostic['frame_count']
            and row['x1'] < diagnostic['width'] and row['y1'] < diagnostic['height']):
        valid_rows.append(row)
valid_rows.sort(key=lambda row: row['detection_id'])
print(f'Valid detections after true-bounds checks: {len(valid_rows)} / {len(candidate_rows)}')
if not valid_rows:
    raise RuntimeError('All candidate detections fall outside actual original bounds.')
selected_row = random.Random(SEED).choice(valid_rows)
info = retriever.get_detection_info(selected_row['detection_id'])
assert isinstance(info, DetectionInfo)
assert isinstance(info.frame_id, str) and isinstance(info.person_id, str)
selected_video = video_by_id[info.video_id]
selected_diagnostic = diagnostics[info.video_id]
display(Markdown(f"## Selected original: {selected_video['title']}"))
display(asdict(info))
display({key: selected_video[key] for key in (
    'video_id', 'filename', 'path', 'duration_seconds', 'fps', 'width', 'height'
)})


## RGB source frame and unresized crop

The expected crop is a diagnostic slice of the public source frame: floor lower bounds, ceil upper bounds, then clamp to decoded dimensions. Exact equality checks that no resizing or pixel changes occurred. Inspect the box and face below for alignment with the visible person.

In [ ]:
source = retriever.get_frame(info.video_id, info.frame_number)
face = retriever.get_face(info.detection_id)
assert source.dtype == np.uint8 and source.ndim == 3 and source.shape[2] == 3
assert source.shape[:2] == (selected_diagnostic['height'], selected_diagnostic['width'])
assert face.dtype == np.uint8 and face.ndim == 3 and face.shape[2] == 3
height, width = source.shape[:2]
x1, y1, x2, y2 = info.bbox
left, top = max(0, min(width, math.floor(x1))), max(0, min(height, math.floor(y1)))
right, bottom = max(0, min(width, math.ceil(x2))), max(0, min(height, math.ceil(y2)))
expected_face = source[top:bottom, left:right]
assert expected_face.size > 0
np.testing.assert_array_equal(face, expected_face)
print(f'Source: shape={source.shape}, dtype={source.dtype}; RGB')
print(f'Crop: width={face.shape[1]}, height={face.shape[0]}, shape={face.shape}, dtype={face.dtype}')
print(f'Exact source-slice comparison: PASS; x=[{left}:{right}], y=[{top}:{bottom}]')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
try:
    axes[0].imshow(source)
    axes[0].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                                fill=False, edgecolor='lime', linewidth=2))
    axes[0].text(x1, y1, f'person {info.person_id}', color='black',
                 bbox=dict(facecolor='lime', alpha=0.85), va='bottom')
    axes[0].set_title(f'Original RGB frame {info.frame_number} (zero-based)')
    axes[1].imshow(face)
    axes[1].set_title(f'Unresized face: {face.shape[1]} × {face.shape[0]} pixels')
    for axis in axes:
        axis.axis('off')
    fig.tight_layout()
    display(fig)
finally:
    plt.close(fig)
del source, face, expected_face, fig, axes, axis


## 32-frame clip contact sheet

The inclusive window is `[reference − 16, reference + 15]`. Each label shows its slot and true original frame index; `PAD` means an out-of-bounds requested index was clamped to the actual first/last frame. Slot **16** is the reference detection, highlighted with a gold border and its box/person label. Array comparisons below check chronology and repetition against fresh `get_frame` requests. Only one clip is retained at a time.

In [ ]:
clip = retriever.get_clip(info.detection_id)
try:
    count = selected_diagnostic['frame_count']
    assert clip.dtype == np.uint8
    assert clip.shape == (32, selected_diagnostic['height'], selected_diagnostic['width'], 3)
    source_indices = [min(max(info.frame_number - 16 + slot, 0), count - 1)
                      for slot in range(32)]
    print(f'Clip: shape={clip.shape}, dtype={clip.dtype}, memory={clip.nbytes / 2**20:.1f} MiB')
    print(f'True original frame count: {count}; reference slot 16 → frame {source_indices[16]}')
    # Sample chronology plus every padded slot; boundary cases later check all 32.
    check_slots = sorted({0, 1, 15, 16, 17, 30, 31} | {
        slot for slot in range(32) if info.frame_number - 16 + slot != source_indices[slot]
    })
    for slot in check_slots:
        comparison = retriever.get_frame(info.video_id, source_indices[slot])
        np.testing.assert_array_equal(clip[slot], comparison)
        del comparison
    print(f'Exact get_frame comparisons PASS for slots: {check_slots}')
    fig, axes = plt.subplots(4, 8, figsize=(24, 12))
    try:
        for slot, axis in enumerate(axes.flat):
            axis.imshow(clip[slot])
            padded = info.frame_number - 16 + slot != source_indices[slot]
            axis.set_title(f"slot {slot} · frame {source_indices[slot]}{' · PAD' if padded else ''}",
                           fontsize=9, color='darkorange' if padded else 'black')
            axis.set_xticks([])
            axis.set_yticks([])
            if slot == 16:
                for spine in axis.spines.values():
                    spine.set_color('gold')
                    spine.set_linewidth(4)
                axis.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                                         fill=False, edgecolor='lime', linewidth=1.5))
                axis.text(x1, y1, f'person {info.person_id}', fontsize=8,
                          bbox=dict(facecolor='lime', alpha=0.85), va='bottom')
        fig.suptitle(f"{selected_video['title']} — detection {info.detection_id}; gold = reference slot 16")
        fig.tight_layout()
        display(fig)
    finally:
        plt.close(fig)
        del fig, axes, axis
finally:
    del clip


## True beginning, interior middle, and end checks

Beginning means a detection at original index `< 16` (actual left padding). End means a detection whose `index + 15 >= decoded_count` (actual right padding). Middle requires a full unpadded window and is chosen nearest the true midpoint. The smallest/largest stored detection alone is not proof of a video boundary.

Every slot of each available case is compared exactly with `get_frame` at its clamped true index. Missing cases are reported as **NOT COVERED**, with no substitute detection or synthetic claim. Short originals can legitimately provide both beginning and end cases but no unpadded middle.

In [ ]:
case_pools = {
    'beginning': [row for row in valid_rows if row['frame_number'] < 16],
    'middle': [row for row in valid_rows if 16 <= row['frame_number']
               and row['frame_number'] + 15 < diagnostics[row['video_id']]['frame_count']],
    'end': [row for row in valid_rows if row['frame_number'] + 15
            >= diagnostics[row['video_id']]['frame_count']],
}
boundary_report = []
for label, pool in case_pools.items():
    if not pool:
        message = f'NOT COVERED: no suitable real {label} detection in the scanned video pool.'
        print(message)
        boundary_report.append(dict(case=label, status=message))
        continue
    def distance_to_target(row):
        n = diagnostics[row['video_id']]['frame_count']
        target = {'beginning': 0, 'middle': (n - 1) / 2, 'end': n - 1}[label]
        return (abs(row['frame_number'] - target), row['video_id'], row['detection_id'])
    case_row = min(pool, key=distance_to_target)
    case_info = retriever.get_detection_info(case_row['detection_id'])
    diagnostic = diagnostics[case_info.video_id]
    n = diagnostic['frame_count']
    clip = retriever.get_clip(case_info.detection_id)
    try:
        assert clip.dtype == np.uint8
        assert clip.shape == (32, diagnostic['height'], diagnostic['width'], 3)
        padded_slots = []
        for slot in range(32):
            requested_index = case_info.frame_number - 16 + slot
            actual_index = min(max(requested_index, 0), n - 1)
            comparison = retriever.get_frame(case_info.video_id, actual_index)
            np.testing.assert_array_equal(clip[slot], comparison,
                                          err_msg=f'{label}: slot {slot}, original {actual_index}')
            del comparison
            if requested_index != actual_index:
                padded_slots.append(slot)
        assert (not padded_slots) if label == 'middle' else bool(padded_slots)
        boundary_report.append(dict(
            case=label, status='PASS: all 32 slots equal original frames',
            detection_id=case_info.detection_id, video_id=case_info.video_id,
            frame_number=case_info.frame_number, true_frame_count=n,
            padded_slots=padded_slots,
        ))
    finally:
        del clip
display(boundary_report)


## Zero-based source timestamps

Compare stored timestamps with actual original-stream PTS at the **enumerated zero-based index**. The report includes raw PTS, PTS relative to decoded frame 0, and the nominal `frame_number / stored_fps`. Source CSV time is elapsed from original frame 0, so relative PTS is the primary check; raw PTS makes a nonzero stream origin visible. Missing PTS is reported, never replaced with inferred timestamps.

Tolerance is half a nominal frame interval plus 1 ms (or 1 ms when neither rate exists). The FPS estimate is a separate diagnostic: variable-frame-rate originals may disagree with it while matching PTS. This checks every eligible stored detection in the pool, including frame 0 when stored. If no stored frame-0 record exists, that gap is explicit. PTS agreement checks timing/index conventions, not face identity.

In [ ]:
timing_rows = []
seen_frames = set()
for row in valid_rows:
    video_id, index = row['video_id'], row['frame_number']
    if (video_id, index) in seen_frames:
        continue
    seen_frames.add((video_id, index))
    diagnostic = diagnostics[video_id]
    fps = video_by_id[video_id]['fps']
    fps = fps if fps is not None and math.isfinite(fps) and fps > 0 else None
    rate = fps or diagnostic['average_rate']
    tolerance = (0.5 / rate if rate and rate > 0 else 0.0) + 0.001
    pts = diagnostic['timestamps'].get(index)
    origin = diagnostic['first_pts']
    relative_pts = pts - origin if pts is not None and origin is not None else None
    stored = row['source_time_seconds']
    pts_error = stored - relative_pts if relative_pts is not None else None
    nominal = index / fps if fps else None
    timing_rows.append(dict(
        video_id=video_id, frame_number=index, stored_seconds=stored,
        original_pts_seconds=pts, original_frame0_pts=origin, relative_pts_seconds=relative_pts,
        nominal_index_over_fps=nominal, tolerance_seconds=tolerance,
        pts_error_seconds=pts_error,
        pts_status=('NO PTS' if pts_error is None else 'PASS' if abs(pts_error) <= tolerance else 'MISMATCH'),
        fps_status=('NO FPS' if nominal is None else 'PASS' if abs(stored - nominal) <= tolerance else 'MISMATCH'),
    ))

for video_id, diagnostic in diagnostics.items():
    zero_rows = [row for row in timing_rows if row['video_id'] == video_id and row['frame_number'] == 0]
    print(f"Video {video_id}: decoded index 0 PTS={diagnostic['first_pts']}; "
          f"decoded index 1 PTS={diagnostic['timestamps'].get(1)}; "
          f"stored frame-0 check={'available' if zero_rows else 'NOT COVERED: no eligible stored frame 0'}")

print(f'Timestamp comparisons: {len(timing_rows)} distinct stored frames.')
display({status: sum(row['pts_status'] == status for row in timing_rows)
         for status in ('PASS', 'MISMATCH', 'NO PTS')})
# Keep the default output readable while showing every mismatch and selected frame.
display([row for row in timing_rows if row['frame_number'] in (0, 1)
         or (row['video_id'], row['frame_number']) == (info.video_id, info.frame_number)
         or row['pts_status'] != 'PASS' or row['fps_status'] == 'MISMATCH'][:50])
print('Full timing report remains available as timing_rows (metadata only).')
assert not any(row['pts_status'] == 'MISMATCH' for row in timing_rows), (
    'Stored source timestamps disagree with original zero-based PTS; inspect timing_rows.'
)


## Review checklist

- Confirm the displayed original title, string person/frame IDs, source dimensions, and RGB colors.
- Inspect the source box and crop for a correctly aligned face; no automated identity claim is made.
- Confirm slot 16 is the detection frame and contact-sheet indices increase chronologically, with repeated indices only at actual edges.
- Read `boundary_report` for real coverage gaps and `timing_rows` for PTS availability / FPS differences. Passing available cases does not imply missing cases were covered.
- Figures are closed after display; each full clip is deleted before the next request. Notebook display outputs still consume kernel/frontend memory until cleared. Restart the kernel or clear outputs after inspection, and keep the saved notebook output-free.

All image retrieval uses the public API. Diagnostic state contains metadata only after these cells complete. No extracted files, persistent caches, resizing, inference, GPU operations, or database updates are performed.